In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox
import sqlite3
from datetime import datetime

# ================= DATABASE =================

conn = sqlite3.connect("classroom_complaints.db")
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS users(
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    username TEXT UNIQUE NOT NULL,
    password TEXT NOT NULL,
    role TEXT NOT NULL)
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS complaints(
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    username TEXT,
    classroom TEXT,
    type TEXT,
    description TEXT,
    date TEXT,
    status TEXT DEFAULT 'Pending')
""")

cur.execute("""
INSERT OR IGNORE INTO users(name,username,password,role)
VALUES('Administrator','admin','admin123','admin')
""")

conn.commit()


# ================= MAIN WINDOW =================

root = tk.Tk()
root.title("Classroom Care - Clean Classroom Complaint System")
root.geometry("900x600")
root.resizable(False, False)
root.configure(bg="#EAF2F8")


# ================= COLORS =================

NAVY = "#17365D"
BLUE = "#2874A6"
LIGHT_BLUE = "#D6EAF8"
GREEN = "#239B56"
RED = "#CB4335"
ORANGE = "#E67E22"
WHITE = "#FFFFFF"
GRAY = "#566573"
BG = "#EAF2F8"


# ================= COMMON FUNCTIONS =================

def clear():
    for w in root.winfo_children():
        w.destroy()


def title_bar(text, subtitle=""):
    frame = tk.Frame(root, bg=NAVY, height=110)
    frame.pack(fill="x")
    frame.pack_propagate(False)

    tk.Label(
        frame,
        text=text,
        font=("Arial", 26, "bold"),
        bg=NAVY,
        fg=WHITE
    ).pack(pady=(18, 2))

    if subtitle:
        tk.Label(
            frame,
            text=subtitle,
            font=("Arial", 11),
            bg=NAVY,
            fg=LIGHT_BLUE
        ).pack()


def footer():
    tk.Label(
        root,
        text="Classroom Care  |  Clean • Report • Improve",
        font=("Arial", 9),
        bg=BG,
        fg=GRAY
    ).pack(side="bottom", pady=8)


def styled_button(parent, text, color, command, width=22):
    return tk.Button(
        parent,
        text=text,
        font=("Arial", 11, "bold"),
        bg=color,
        fg=WHITE,
        activebackground=color,
        activeforeground=WHITE,
        width=width,
        height=2,
        relief="flat",
        cursor="hand2",
        command=command
    )


# ================= HOME =================

def home():
    clear()

    # Header
    header = tk.Frame(root, bg=NAVY, height=150)
    header.pack(fill="x")
    header.pack_propagate(False)

    tk.Label(
        header,
        text="🏫 Classroom Care",
        font=("Arial", 30, "bold"),
        bg=NAVY,
        fg=WHITE
    ).pack(pady=(22, 5))

    tk.Label(
        header,
        text="Classroom Cleanliness Complaint Management System",
        font=("Arial", 12),
        bg=NAVY,
        fg=LIGHT_BLUE
    ).pack()

    # Welcome card
    card = tk.Frame(root, bg=WHITE, width=620, height=260)
    card.pack(pady=25)
    card.pack_propagate(False)

    tk.Label(
        card,
        text="Welcome!",
        font=("Arial", 22, "bold"),
        bg=WHITE,
        fg=NAVY
    ).pack(pady=(20, 5))

    tk.Label(
        card,
        text="Report classroom cleanliness problems easily\n"
             "and help maintain a cleaner learning environment.",
        font=("Arial", 11),
        bg=WHITE,
        fg=GRAY,
        justify="center"
    ).pack(pady=5)

    styled_button(
        card,
        "🔐  Login",
        BLUE,
        login_page,
        28
    ).pack(pady=(15, 8))

    styled_button(
        card,
        "📝  Student Registration",
        GREEN,
        register_page,
        28
    ).pack()

    styled_button(
        root,
        "Exit",
        RED,
        root.destroy,
        12
    ).pack(pady=5)

    footer()


# ================= REGISTRATION =================

def register_page():
    clear()

    title_bar(
        "📝 Student Registration",
        "Create your Classroom Care account"
    )

    card = tk.Frame(root, bg=WHITE, width=550, height=390)
    card.pack(pady=20)
    card.pack_propagate(False)

    entries = {}

    for text in ["Name", "Username", "Password"]:

        tk.Label(
            card,
            text=text,
            font=("Arial", 11, "bold"),
            bg=WHITE,
            fg=NAVY
        ).pack(pady=(18, 4))

        e = tk.Entry(
            card,
            width=38,
            font=("Arial", 11),
            relief="solid",
            bd=1
        )
        e.pack(ipady=5)

        if text == "Password":
            e.config(show="*")

        entries[text] = e

    def register():

        name = entries["Name"].get().strip()
        user = entries["Username"].get().strip()
        password = entries["Password"].get().strip()

        if not name or not user or not password:
            messagebox.showwarning(
                "Warning",
                "Please fill all fields."
            )
            return

        try:
            cur.execute(
                """
                INSERT INTO users(name,username,password,role)
                VALUES(?,?,?,'student')
                """,
                (name, user, password)
            )

            conn.commit()

            messagebox.showinfo(
                "Success",
                "Registration successful!"
            )

            login_page()

        except sqlite3.IntegrityError:
            messagebox.showerror(
                "Error",
                "Username already exists."
            )

    styled_button(
        card,
        "Create Account",
        GREEN,
        register,
        20
    ).pack(pady=18)

    styled_button(
        card,
        "Back",
        GRAY,
        home,
        12
    ).pack()


# ================= LOGIN =================

def login_page():
    clear()

    title_bar(
        "🔐 Login",
        "Access your Classroom Care dashboard"
    )

    card = tk.Frame(root, bg=WHITE, width=550, height=390)
    card.pack(pady=20)
    card.pack_propagate(False)

    tk.Label(
        card,
        text="Username",
        font=("Arial", 11, "bold"),
        bg=WHITE,
        fg=NAVY
    ).pack(pady=(25, 5))

    user = tk.Entry(
        card,
        width=38,
        font=("Arial", 11),
        relief="solid",
        bd=1
    )
    user.pack(ipady=5)

    tk.Label(
        card,
        text="Password",
        font=("Arial", 11, "bold"),
        bg=WHITE,
        fg=NAVY
    ).pack(pady=(18, 5))

    password = tk.Entry(
        card,
        width=38,
        font=("Arial", 11),
        show="*",
        relief="solid",
        bd=1
    )
    password.pack(ipady=5)

    role = tk.StringVar(value="student")

    role_frame = tk.Frame(card, bg=WHITE)
    role_frame.pack(pady=15)

    tk.Label(
        role_frame,
        text="Login as:",
        font=("Arial", 10, "bold"),
        bg=WHITE,
        fg=GRAY
    ).pack(side="left", padx=8)

    tk.Radiobutton(
        role_frame,
        text="Student",
        variable=role,
        value="student",
        bg=WHITE,
        activebackground=WHITE
    ).pack(side="left")

    tk.Radiobutton(
        role_frame,
        text="Admin",
        variable=role,
        value="admin",
        bg=WHITE,
        activebackground=WHITE
    ).pack(side="left")

    def login():

        u = user.get().strip()
        p = password.get().strip()
        r = role.get()

        cur.execute(
            """
            SELECT * FROM users
            WHERE username=? AND password=? AND role=?
            """,
            (u, p, r)
        )

        data = cur.fetchone()

        if data:
            if r == "student":
                student_dashboard(u)
            else:
                admin_dashboard()
        else:
            messagebox.showerror(
                "Login Failed",
                "Invalid username, password or role."
            )

    styled_button(
        card,
        "Login",
        BLUE,
        login,
        20
    ).pack(pady=10)

    styled_button(
        card,
        "Back",
        GRAY,
        home,
        12
    ).pack()


# ================= STUDENT DASHBOARD =================

def student_dashboard(username):
    clear()

    title_bar(
        "🎓 Student Dashboard",
        "Manage your classroom cleanliness complaints"
    )

    tk.Label(
        root,
        text="Welcome, " + username + " 👋",
        font=("Arial", 16, "bold"),
        bg=BG,
        fg=NAVY
    ).pack(pady=20)

    card = tk.Frame(root, bg=WHITE, width=650, height=260)
    card.pack()
    card.pack_propagate(False)

    tk.Label(
        card,
        text="What would you like to do?",
        font=("Arial", 16, "bold"),
        bg=WHITE,
        fg=NAVY
    ).pack(pady=20)

    styled_button(
        card,
        "📝  Submit Complaint",
        BLUE,
        lambda: complaint_page(username),
        30
    ).pack(pady=8)

    styled_button(
        card,
        "📋  My Complaints",
        GREEN,
        lambda: my_complaints(username),
        30
    ).pack(pady=8)

    styled_button(
        root,
        "Logout",
        RED,
        home,
        15
    ).pack(pady=18)


# ================= SUBMIT COMPLAINT =================

def complaint_page(username):
    clear()

    title_bar(
        "📝 Submit Complaint",
        "Report a classroom cleanliness problem"
    )

    card = tk.Frame(root, bg=WHITE, width=650, height=420)
    card.pack(pady=15)
    card.pack_propagate(False)

    tk.Label(
        card,
        text="Classroom",
        font=("Arial", 11, "bold"),
        bg=WHITE,
        fg=NAVY
    ).pack(pady=(18, 4))

    room = ttk.Combobox(
        card,
        values=[
            "Room 101",
            "Room 102",
            "Room 103",
            "Room 104",
            "Computer Lab",
            "Electronics Lab"
        ],
        state="readonly",
        width=40
    )
    room.pack()

    tk.Label(
        card,
        text="Complaint Type",
        font=("Arial", 11, "bold"),
        bg=WHITE,
        fg=NAVY
    ).pack(pady=(12, 4))

    typ = ttk.Combobox(
        card,
        values=[
            "Dirty Floor",
            "Dustbin Problem",
            "Dirty Desk",
            "Bad Odour",
            "Spillage",
            "Washroom Cleanliness",
            "Other"
        ],
        state="readonly",
        width=40
    )
    typ.pack()

    tk.Label(
        card,
        text="Description",
        font=("Arial", 11, "bold"),
        bg=WHITE,
        fg=NAVY
    ).pack(pady=(12, 4))

    desc = tk.Text(
        card,
        width=48,
        height=5,
        font=("Arial", 10),
        relief="solid",
        bd=1
    )
    desc.pack()

    def submit():

        if (
            not room.get()
            or not typ.get()
            or not desc.get("1.0", "end").strip()
        ):
            messagebox.showwarning(
                "Warning",
                "Please fill all fields."
            )
            return

        cur.execute(
            """
            INSERT INTO complaints
            (username,classroom,type,description,date,status)
            VALUES(?,?,?,?,?,?)
            """,
            (
                username,
                room.get(),
                typ.get(),
                desc.get("1.0", "end").strip(),
                datetime.now().strftime("%Y-%m-%d %H:%M"),
                "Pending"
            )
        )

        conn.commit()

        messagebox.showinfo(
            "Success",
            "Complaint submitted successfully!"
        )

        student_dashboard(username)

    frame = tk.Frame(card, bg=WHITE)
    frame.pack(pady=12)

    styled_button(
        frame,
        "Submit",
        GREEN,
        submit,
        15
    ).pack(side="left", padx=8)

    styled_button(
        frame,
        "Back",
        GRAY,
        lambda: student_dashboard(username),
        12
    ).pack(side="left", padx=8)


# ================= MY COMPLAINTS =================

def my_complaints(username):
    clear()

    title_bar(
        "📋 My Complaints",
        "View the status of your submitted complaints"
    )

    style = ttk.Style()
    style.theme_use("clam")

    style.configure(
        "Treeview",
        font=("Arial", 10),
        rowheight=30,
        background="white",
        fieldbackground="white"
    )

    style.configure(
        "Treeview.Heading",
        font=("Arial", 10, "bold"),
        background=NAVY,
        foreground=WHITE
    )

    columns = (
        "ID",
        "Classroom",
        "Type",
        "Description",
        "Date",
        "Status"
    )

    tree = ttk.Treeview(
        root,
        columns=columns,
        show="headings",
        height=12
    )

    for c in columns:
        tree.heading(c, text=c)

    tree.column("ID", width=50)
    tree.column("Classroom", width=100)
    tree.column("Type", width=130)
    tree.column("Description", width=250)
    tree.column("Date", width=120)
    tree.column("Status", width=100)

    tree.pack(pady=15)

    cur.execute(
        """
        SELECT id,classroom,type,description,date,status
        FROM complaints
        WHERE username=?
        """,
        (username,)
    )

    for row in cur.fetchall():
        tree.insert("", tk.END, values=row)

    styled_button(
        root,
        "Back",
        GRAY,
        lambda: student_dashboard(username),
        15
    ).pack(pady=8)


# ================= ADMIN DASHBOARD =================

def admin_dashboard():
    clear()

    title_bar(
        "🛠️ Admin Dashboard",
        "Monitor and manage classroom complaints"
    )

    card = tk.Frame(root, bg=WHITE, width=650, height=280)
    card.pack(pady=25)
    card.pack_propagate(False)

    tk.Label(
        card,
        text="Administrator Panel",
        font=("Arial", 18, "bold"),
        bg=WHITE,
        fg=NAVY
    ).pack(pady=25)

    styled_button(
        card,
        "📋  View All Complaints",
        BLUE,
        all_complaints,
        30
    ).pack(pady=10)

    styled_button(
        card,
        "📊  Statistics",
        ORANGE,
        statistics,
        30
    ).pack(pady=10)

    styled_button(
        root,
        "Logout",
        RED,
        home,
        15
    ).pack(pady=10)


# ================= ALL COMPLAINTS =================

def all_complaints():
    clear()

    title_bar(
        "📋 All Complaints",
        "Review and update complaint status"
    )

    style = ttk.Style()
    style.theme_use("clam")

    style.configure(
        "Treeview",
        font=("Arial", 9),
        rowheight=30,
        background="white",
        fieldbackground="white"
    )

    style.configure(
        "Treeview.Heading",
        font=("Arial", 9, "bold"),
        background=NAVY,
        foreground=WHITE
    )

    columns = (
        "ID",
        "User",
        "Room",
        "Type",
        "Description",
        "Date",
        "Status"
    )

    tree = ttk.Treeview(
        root,
        columns=columns,
        show="headings",
        height=10
    )

    for c in columns:
        tree.heading(c, text=c)

    widths = [40, 80, 80, 110, 230, 120, 90]

    for c, w in zip(columns, widths):
        tree.column(c, width=w)

    tree.pack(pady=10)

    def load():

        for item in tree.get_children():
            tree.delete(item)

        cur.execute("SELECT * FROM complaints")

        for row in cur.fetchall():
            tree.insert("", tk.END, values=row)

    load()

    frame = tk.Frame(root, bg=BG)
    frame.pack(pady=8)

    tk.Label(
        frame,
        text="Change Status:",
        font=("Arial", 10, "bold"),
        bg=BG,
        fg=NAVY
    ).pack(side="left", padx=5)

    status = ttk.Combobox(
        frame,
        values=[
            "Pending",
            "In Progress",
            "Resolved"
        ],
        state="readonly",
        width=15
    )
    status.pack(side="left", padx=5)

    def update():

        selected = tree.selection()

        if not selected:
            messagebox.showwarning(
                "Warning",
                "Please select a complaint."
            )
            return

        if not status.get():
            messagebox.showwarning(
                "Warning",
                "Please select a status."
            )
            return

        complaint_id = tree.item(
            selected[0]
        )["values"][0]

        cur.execute(
            "UPDATE complaints SET status=? WHERE id=?",
            (status.get(), complaint_id)
        )

        conn.commit()

        load()

        messagebox.showinfo(
            "Success",
            "Complaint status updated!"
        )

    styled_button(
        frame,
        "Update Status",
        ORANGE,
        update,
        15
    ).pack(side="left", padx=5)

    styled_button(
        root,
        "Back",
        GRAY,
        admin_dashboard,
        15
    ).pack(pady=5)


# ================= STATISTICS =================

def statistics():
    clear()

    title_bar(
        "📊 Complaint Statistics",
        "Overview of classroom complaints"
    )

    cur.execute("SELECT COUNT(*) FROM complaints")
    total = cur.fetchone()[0]

    cur.execute(
        "SELECT COUNT(*) FROM complaints WHERE status='Pending'"
    )
    pending = cur.fetchone()[0]

    cur.execute(
        "SELECT COUNT(*) FROM complaints WHERE status='In Progress'"
    )
    progress = cur.fetchone()[0]

    cur.execute(
        "SELECT COUNT(*) FROM complaints WHERE status='Resolved'"
    )
    resolved = cur.fetchone()[0]

    # Statistics cards
    container = tk.Frame(root, bg=BG)
    container.pack(pady=35)

    def stat_card(parent, title, value, color):
        card = tk.Frame(
            parent,
            bg=WHITE,
            width=180,
            height=150
        )
        card.pack(side="left", padx=10)
        card.pack_propagate(False)

        tk.Label(
            card,
            text=title,
            font=("Arial", 11, "bold"),
            bg=WHITE,
            fg=GRAY
        ).pack(pady=(20, 8))

        tk.Label(
            card,
            text=str(value),
            font=("Arial", 28, "bold"),
            bg=WHITE,
            fg=color
        ).pack()

    stat_card(container, "TOTAL", total, NAVY)
    stat_card(container, "PENDING", pending, ORANGE)
    stat_card(container, "IN PROGRESS", progress, BLUE)
    stat_card(container, "RESOLVED", resolved, GREEN)

    tk.Label(
        root,
        text="Complaint status summary",
        font=("Arial", 11),
        bg=BG,
        fg=GRAY
    ).pack(pady=10)

    styled_button(
        root,
        "Back",
        GRAY,
        admin_dashboard,
        15
    ).pack(pady=15)


# ================= START =================

home()
root.mainloop()

In [3]:
import os
import sqlite3

# Create a fresh database
if os.path.exists("classroom_complaints.db"):
    os.remove("classroom_complaints.db")

conn = sqlite3.connect("classroom_complaints.db")
cur = conn.cursor()

cur.execute("""
CREATE TABLE users(
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    username TEXT UNIQUE NOT NULL,
    password TEXT NOT NULL,
    role TEXT NOT NULL
)
""")

cur.execute("""
CREATE TABLE complaints(
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    username TEXT,
    classroom TEXT,
    type TEXT,
    description TEXT,
    date TEXT,
    status TEXT DEFAULT 'Pending'
)
""")

cur.execute("""
INSERT INTO users(name, username, password, role)
VALUES('Administrator', 'admin', 'admin123', 'admin')
""")

conn.commit()

print("Database created successfully!")
print("\nComplaints table columns:")
print(cur.execute("PRAGMA table_info(complaints)").fetchall())

PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'classroom_complaints.db'